In [4]:
import pandas as pd

file_path = "/Users/aasiyaahmedalmani/Downloads/yelp_merged_dataset.json"

chunk_size = 100000  # 🔽 smaller chunk = safer
chunks = []

print("=== Reading and combining dataset ===")

for i, chunk in enumerate(pd.read_json(file_path, lines=True, chunksize=chunk_size)):
    print(f"📥 Chunk {i+1} loaded")

    # Optional: reduce memory (IMPORTANT)
    chunk = chunk.dropna(subset=['user_id', 'business_id', 'stars_x', 'date'])

    chunks.append(chunk)

print("🔄 Combining all chunks...")

df = pd.concat(chunks, ignore_index=True)

print("\n✅ Done")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

=== Reading and combining dataset ===
📥 Chunk 1 loaded
📥 Chunk 2 loaded
📥 Chunk 3 loaded
📥 Chunk 4 loaded
📥 Chunk 5 loaded
📥 Chunk 6 loaded
📥 Chunk 7 loaded
📥 Chunk 8 loaded
📥 Chunk 9 loaded
📥 Chunk 10 loaded
📥 Chunk 11 loaded
📥 Chunk 12 loaded
📥 Chunk 13 loaded
📥 Chunk 14 loaded
📥 Chunk 15 loaded
📥 Chunk 16 loaded
📥 Chunk 17 loaded
📥 Chunk 18 loaded
📥 Chunk 19 loaded
📥 Chunk 20 loaded
📥 Chunk 21 loaded
📥 Chunk 22 loaded
📥 Chunk 23 loaded
📥 Chunk 24 loaded
📥 Chunk 25 loaded
📥 Chunk 26 loaded
📥 Chunk 27 loaded
📥 Chunk 28 loaded
📥 Chunk 29 loaded
📥 Chunk 30 loaded
📥 Chunk 31 loaded
📥 Chunk 32 loaded
📥 Chunk 33 loaded
📥 Chunk 34 loaded
📥 Chunk 35 loaded
📥 Chunk 36 loaded
📥 Chunk 37 loaded
📥 Chunk 38 loaded
📥 Chunk 39 loaded
📥 Chunk 40 loaded
📥 Chunk 41 loaded
📥 Chunk 42 loaded
📥 Chunk 43 loaded
📥 Chunk 44 loaded
📥 Chunk 45 loaded
📥 Chunk 46 loaded
📥 Chunk 47 loaded
📥 Chunk 48 loaded
📥 Chunk 49 loaded
📥 Chunk 50 loaded
📥 Chunk 51 loaded
📥 Chunk 52 loaded
📥 Chunk 53 loaded
📥 Chunk 54 loaded

In [5]:
import pandas as pd

alerts = pd.read_csv("/Users/aasiyaahmedalmani/Downloads/yelp_consumer_alerts.csv")

print(alerts.shape)
print(alerts.head())

(6608, 6)
              Business Name              Location           Date  \
0  Zak Dental - Simi Valley       Simi Valley, CA  February 2026   
1                    WeVINO  Westlake Village, CA  February 2026   
2                   Ume Tea           Fremont, CA  February 2026   
3        Toast Cafe & Grill          San Jose, CA  February 2026   
4            Tin Roof Grill             Sandy, UT  February 2026   

                                        Business URL  Evidence URL  \
0  https://www.yelp.com/biz/zak-dental-simi-valle...           NaN   
1   https://www.yelp.com/biz/wevino-westlake-village           NaN   
2         https://www.yelp.com/biz/ume-tea-fremont-7           NaN   
3  https://www.yelp.com/biz/toast-cafe-and-grill-...           NaN   
4      https://www.yelp.com/biz/tin-roof-grill-sandy           NaN   

                             Alert Type  
0  Previous Compensated Activity Alerts  
1  Previous Compensated Activity Alerts  
2  Previous Compensated Activity A

In [6]:
# Clean business name
alerts['name_clean'] = alerts['Business Name'].str.lower().str.strip()

# Split location safely
alerts[['city', 'state']] = alerts['Location'].str.split(',', n=1, expand=True)

# Clean values
alerts['city'] = alerts['city'].str.lower().str.strip()
alerts['state'] = alerts['state'].str.strip()

# Drop missing (this was causing your error earlier)
alerts = alerts.dropna(subset=['city', 'state'])

# Optional: remove duplicates
alerts = alerts.drop_duplicates(subset=['name_clean', 'city', 'state'])

# Check
print(alerts[['Business Name','Location','city','state']].head())
print("Shape:", alerts.shape)

              Business Name              Location              city state
0  Zak Dental - Simi Valley       Simi Valley, CA       simi valley    CA
1                    WeVINO  Westlake Village, CA  westlake village    CA
2                   Ume Tea           Fremont, CA           fremont    CA
3        Toast Cafe & Grill          San Jose, CA          san jose    CA
4            Tin Roof Grill             Sandy, UT             sandy    UT
Shape: (6132, 9)


In [7]:
df['name_clean'] = df['name'].str.lower().str.strip()
df['city'] = df['city'].str.lower().str.strip()
df['state'] = df['state'].str.strip()

In [8]:
matched = df.merge(
    alerts[['name_clean', 'city', 'state']],
    on=['name_clean', 'city', 'state'],
    how='inner'
)

In [9]:
print(matched[['name','city','state']].head())

         name          city state
0  Twin Peaks  indianapolis    IN
1  Twin Peaks  indianapolis    IN
2  Gator Ford       seffner    FL
3  Twin Peaks  indianapolis    IN
4  Twin Peaks  indianapolis    IN


In [10]:
matched_users = matched['user_id'].nunique()
total_users = df['user_id'].nunique()

print("Total users:", total_users)
print("Users who reviewed flagged businesses:", matched_users)

print("Percentage:", round((matched_users / total_users)*100, 2), "%")

Total users: 1987929
Users who reviewed flagged businesses: 9769
Percentage: 0.49 %


In [11]:
flagged_users = set(matched['user_id'].unique())

print("Flagged users:", len(flagged_users))

Flagged users: 9769


In [13]:
import pandas as pd

input_path = "/Users/aasiyaahmedalmani/Downloads/yelp_merged_dataset.json"
output_path = "/Users/aasiyaahmedalmani/Downloads/yelp_cleanedsuspicious_users_removed_new.json"

chunk_size = 200000

print("=== Removing flagged users and saving new file ===")

# Open file in write mode first (clears file)
open(output_path, 'w').close()

for i, chunk in enumerate(pd.read_json(input_path, lines=True, chunksize=chunk_size)):
    print(f"📥 Processing chunk {i+1}")

    # Remove flagged users
    chunk_clean = chunk[~chunk['user_id'].isin(flagged_users)]

    # Append to new file
    chunk_clean.to_json(
        output_path,
        orient='records',
        lines=True,
        mode='a'
    )

print("\n✅ Cleaned dataset saved successfully!")

=== Removing flagged users and saving new file ===
📥 Processing chunk 1
📥 Processing chunk 2
📥 Processing chunk 3
📥 Processing chunk 4
📥 Processing chunk 5
📥 Processing chunk 6
📥 Processing chunk 7
📥 Processing chunk 8
📥 Processing chunk 9
📥 Processing chunk 10
📥 Processing chunk 11
📥 Processing chunk 12
📥 Processing chunk 13
📥 Processing chunk 14
📥 Processing chunk 15
📥 Processing chunk 16
📥 Processing chunk 17
📥 Processing chunk 18
📥 Processing chunk 19
📥 Processing chunk 20
📥 Processing chunk 21
📥 Processing chunk 22
📥 Processing chunk 23
📥 Processing chunk 24
📥 Processing chunk 25
📥 Processing chunk 26
📥 Processing chunk 27
📥 Processing chunk 28
📥 Processing chunk 29
📥 Processing chunk 30
📥 Processing chunk 31
📥 Processing chunk 32
📥 Processing chunk 33
📥 Processing chunk 34
📥 Processing chunk 35

✅ Cleaned dataset saved successfully!


In [14]:
count = 0

for chunk in pd.read_json(output_path, lines=True, chunksize=200000):
    count += len(chunk)

print("Rows in cleaned dataset:", count)

Rows in cleaned dataset: 6740564


In [16]:
import pandas as pd

file_path = "/Users/aasiyaahmedalmani/Downloads/yelp_cleanedsuspicious_users_removed_new.json"

chunk_size = 100000   # smaller = safer
chunks = []

print("=== Reading and combining dataset ===")

for i, chunk in enumerate(pd.read_json(file_path, lines=True, chunksize=chunk_size)):
    print(f"📥 Chunk {i+1}")

    # Optional: reduce memory
    chunk = chunk[['user_id','business_id','stars_x','date']]

    chunks.append(chunk)

print("🔄 Combining...")

df = pd.concat(chunks, ignore_index=True)

print("\n✅ Done")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

=== Reading and combining dataset ===
📥 Chunk 1
📥 Chunk 2
📥 Chunk 3
📥 Chunk 4
📥 Chunk 5
📥 Chunk 6
📥 Chunk 7
📥 Chunk 8
📥 Chunk 9
📥 Chunk 10
📥 Chunk 11
📥 Chunk 12
📥 Chunk 13
📥 Chunk 14
📥 Chunk 15
📥 Chunk 16
📥 Chunk 17
📥 Chunk 18
📥 Chunk 19
📥 Chunk 20
📥 Chunk 21
📥 Chunk 22
📥 Chunk 23
📥 Chunk 24
📥 Chunk 25
📥 Chunk 26
📥 Chunk 27
📥 Chunk 28
📥 Chunk 29
📥 Chunk 30
📥 Chunk 31
📥 Chunk 32
📥 Chunk 33
📥 Chunk 34
📥 Chunk 35
📥 Chunk 36
📥 Chunk 37
📥 Chunk 38
📥 Chunk 39
📥 Chunk 40
📥 Chunk 41
📥 Chunk 42
📥 Chunk 43
📥 Chunk 44
📥 Chunk 45
📥 Chunk 46
📥 Chunk 47
📥 Chunk 48
📥 Chunk 49
📥 Chunk 50
📥 Chunk 51
📥 Chunk 52
📥 Chunk 53
📥 Chunk 54
📥 Chunk 55
📥 Chunk 56
📥 Chunk 57
📥 Chunk 58
📥 Chunk 59
📥 Chunk 60
📥 Chunk 61
📥 Chunk 62
📥 Chunk 63
📥 Chunk 64
📥 Chunk 65
📥 Chunk 66
📥 Chunk 67
📥 Chunk 68
🔄 Combining...

✅ Done
Shape: (6740564, 4)
Columns: ['user_id', 'business_id', 'stars_x', 'date']


In [17]:
df = df.sort_values(['user_id','date']).copy()

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.dropna(subset=['date'])

In [18]:
# ==========================================
# USER-LEVEL CONTROLS
# ==========================================

# Total number of reviews written by each user
user_controls = df.groupby('user_id').agg(
    total_reviews=('date', 'count'),
    first_review=('date', 'min')
).reset_index()

# Time since first review (in months)
analysis_end = df['date'].max()

user_controls['months_since_first_review'] = (
    (analysis_end - user_controls['first_review']).dt.days / 30.44
)

# Avoid division by zero
user_controls['months_since_first_review'] = (
    user_controls['months_since_first_review']
    .clip(lower=1)
)

print(user_controls.head())

                  user_id  total_reviews        first_review  \
0  ---1lKK3aKOuomHnwAkAow              1 2018-12-19 22:26:22   
1  ---2PmXbF47D870stH1jqA             28 2012-10-24 13:33:39   
2  ---UgP94gokyCDuB5zUssA             11 2014-11-02 15:10:18   
3  ---fa6ZK37T9NjkGKI4oSg              1 2019-04-08 21:18:12   
4  ---r61b7EpVPkb4UVme5tA              5 2014-04-28 17:03:59   

   months_since_first_review  
0                  36.990802  
1                 110.840999  
2                  86.563732  
3                  33.377135  
4                  92.739816  


In [19]:
df['day'] = df['date'].dt.date

day_counts = df.groupby(['user_id','day']).size().reset_index(name='n_reviews')

valid_days = day_counts[day_counts['n_reviews'] <= 30]

df = df.merge(valid_days[['user_id','day']], on=['user_id','day'], how='inner')

In [20]:
df['prev_date'] = df.groupby('user_id')['date'].shift(1)

df['gap_min'] = (df['date'] - df['prev_date']).dt.total_seconds() / 60

df['bundled_session_id'] = (
    (df['gap_min'] > 10) | (df['gap_min'].isna())
).groupby(df['user_id']).cumsum()

In [21]:
# RECOMPUTE GAP (important after filtering)
df = df.sort_values(['user_id','date']).copy()

df['gap_min'] = df.groupby('user_id')['date'].diff().dt.total_seconds() / 60

# CORRECT LOGIC:
# New session starts when gap <= 60 (NOT >60)
df['separate_session_id'] = (
    df['gap_min'].isna() | (df['gap_min'] <= 60)
).groupby(df['user_id']).cumsum()

In [22]:
bundled_sizes = df.groupby(['user_id','bundled_session_id']).size().reset_index(name='n_reviews')

valid_bundled = bundled_sizes[bundled_sizes['n_reviews'] >= 5]

df_bundled = df.merge(valid_bundled, on=['user_id','bundled_session_id'], how='inner')
df_bundled['session_type'] = 'bundled'

In [23]:
sep_stats = df.groupby(['user_id','separate_session_id']).agg(
    session_start=('date','min'),
    session_end=('date','max'),
    n_reviews=('date','count')
).reset_index()

# CORRECT duration check (not .dt.days ❌)
valid_separate = sep_stats[
    (sep_stats['n_reviews'] >= 5) &
    ((sep_stats['session_end'] - sep_stats['session_start']) <= pd.Timedelta(days=7))
]

df_separate = df.merge(
    valid_separate[['user_id','separate_session_id']],
    on=['user_id','separate_session_id'],
    how='inner'
)

df_separate['session_type'] = 'separate'

In [24]:
df_sessions = pd.concat([df_bundled, df_separate], ignore_index=True)

print("✅ Final sessions dataset shape:", df_sessions.shape)

✅ Final sessions dataset shape: (256779, 11)


In [25]:
print(df_sessions['session_type'].value_counts())

print(df_sessions.groupby('session_type')['stars_x'].mean())

session_type
bundled     234677
separate     22102
Name: count, dtype: int64
session_type
bundled     3.996685
separate    3.962764
Name: stars_x, dtype: float64


In [26]:
# CORRECT session IDs

# Bundled
df_bundled['session_id'] = (
    df_bundled['user_id'].astype(str) + "_B_" +
    df_bundled['bundled_session_id'].astype(str)
)

# Separate
df_separate['session_id'] = (
    df_separate['user_id'].astype(str) + "_S_" +
    df_separate['separate_session_id'].astype(str)
)

# Combine AFTER creating IDs
df_sessions = pd.concat([df_bundled, df_separate], ignore_index=True)

In [27]:
print(df_sessions.columns)

Index(['user_id', 'business_id', 'stars_x', 'date', 'day', 'prev_date',
       'gap_min', 'bundled_session_id', 'separate_session_id', 'n_reviews',
       'session_type', 'session_id'],
      dtype='object')


In [28]:
user_behavior = df_sessions.groupby('user_id')['session_type'].agg(
    lambda x: set(x)
).reset_index()

user_behavior.columns = ['user_id', 'session_types']

In [29]:
def classify_user(x):
    if x == {'bundled'}:
        return 'only_bundled'
    elif x == {'separate'}:
        return 'only_separate'
    else:
        return 'both'

user_behavior['user_group'] = user_behavior['session_types'].apply(classify_user)

In [30]:
user_counts = user_behavior['user_group'].value_counts()

print(user_counts)

user_group
only_bundled     26010
only_separate     2313
both               706
Name: count, dtype: int64


In [31]:
user_percent = user_behavior['user_group'].value_counts(normalize=True) * 100

print(user_percent)

user_group
only_bundled     89.600055
only_separate     7.967894
both              2.432051
Name: proportion, dtype: float64


In [32]:
df_sessions = df_sessions.merge(
    user_behavior[['user_id','user_group']],
    on='user_id',
    how='left'
)

print(df_sessions.groupby('user_group')['session_id'].nunique())

user_group
both              3138
only_bundled     34285
only_separate     2719
Name: session_id, dtype: int64


In [33]:
print(df_sessions.groupby('user_group')['stars_x'].mean())

user_group
both             3.833735
only_bundled     4.006548
only_separate    4.012355
Name: stars_x, dtype: float64


In [34]:
dup_check = df_sessions.duplicated(
    subset=['user_id','business_id','date'],
    keep=False
)

print("Overlapping rows:", dup_check.sum())

Overlapping rows: 284


In [35]:
session_stats = df_sessions.groupby('session_id').agg(
    
    user_id=('user_id','first'),
    session_type=('session_type','first'),
    
    # ⭐ MAIN OUTCOME VARIABLES
    mean_rating=('stars_x','mean'),
    var_rating=('stars_x','var'),
    sd_rating=('stars_x','std'),
    unique_ratings=('stars_x','nunique'),
    
    # 📊 SESSION CHARACTERISTICS
    n_reviews=('stars_x','count'),
    min_rating=('stars_x','min'),
    max_rating=('stars_x','max'),
    
    # ⏱ TIME
    session_start=('date','min'),
    session_end=('date','max')
    
).reset_index()

In [36]:
print("Variance:")
print(session_stats.groupby('session_type')['var_rating'].mean())

print("\nMean rating:")
print(session_stats.groupby('session_type')['mean_rating'].mean())

print("\nUnique ratings:")
print(session_stats.groupby('session_type')['unique_ratings'].mean())

Variance:
session_type
bundled     1.008016
separate    0.976046
Name: var_rating, dtype: float64

Mean rating:
session_type
bundled     4.002387
separate    3.963504
Name: mean_rating, dtype: float64

Unique ratings:
session_type
bundled     2.805586
separate    2.752174
Name: unique_ratings, dtype: float64


In [37]:
from linearmodels.panel import PanelOLS

reg_df = session_stats.copy()

# =========================
# IV: is_bundled
# =========================
reg_df['is_bundled'] = (reg_df['session_type'] == 'bundled').astype(int)

# =========================
# IV: weekend
# =========================
reg_df['weekend'] = reg_df['session_start'].dt.dayofweek.isin([5,6]).astype(int)

# =========================
# IV: hour fixed effect
# =========================
reg_df['hour_fe'] = reg_df['session_start'].dt.hour

# =========================
# PANEL INDEX (REQUIRED)
# =========================
reg_df['session_num'] = reg_df.groupby('user_id').cumcount()

reg_df = reg_df.set_index(['user_id','session_num'])

In [38]:
# ==========================================
# MERGE USER-LEVEL CONTROLS
# ==========================================

reg_df = reg_df.reset_index()

reg_df = reg_df.merge(
    user_controls[['user_id',
                   'total_reviews',
                   'months_since_first_review']],
    on='user_id',
    how='left'
)

reg_df = reg_df.set_index(['user_id', 'session_num'])

print(reg_df.columns)

Index(['session_id', 'session_type', 'mean_rating', 'var_rating', 'sd_rating',
       'unique_ratings', 'n_reviews', 'min_rating', 'max_rating',
       'session_start', 'session_end', 'is_bundled', 'weekend', 'hour_fe',
       'total_reviews', 'months_since_first_review'],
      dtype='object')


In [39]:
model_var = PanelOLS.from_formula(
    """
    var_rating ~
    is_bundled +
    n_reviews +
    total_reviews +
    months_since_first_review +
    weekend +
    hour_fe +
    EntityEffects
    """,
    data=reg_df,
    drop_absorbed=True
)

result_var = model_var.fit(cov_type="robust")
print(result_var.summary)

/var/folders/n_/7ftmcsrs6ng_hzblgzjhb9040000gn/T/ipykernel_29802/3351961077.py:16: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

total_reviews, months_since_first_review

  result_var = model_var.fit(cov_type="robust")


                          PanelOLS Estimation Summary                           
Dep. Variable:             var_rating   R-squared:                        0.0002
Estimator:                   PanelOLS   R-squared (Between):             -0.0262
No. Observations:               40142   R-squared (Within):               0.0002
Date:                Tue, Jun 30 2026   R-squared (Overall):             -0.0259
Time:                        12:55:17   Log-likelihood                -1.521e+04
Cov. Estimator:                Robust                                           
                                        F-statistic:                      0.6692
Entities:                       29029   P-value                           0.6133
Avg Obs:                       1.3828   Distribution:                 F(4,11109)
Min Obs:                       1.0000                                           
Max Obs:                       37.000   F-statistic (robust):             0.2655
                            

In [41]:
model_unique = PanelOLS.from_formula(
    """
    unique_ratings ~
    is_bundled +
    n_reviews +
    total_reviews +
    months_since_first_review +
    weekend +
    hour_fe +
    EntityEffects
    """,
    data=reg_df,
    drop_absorbed=True
)

result_unique = model_unique.fit(cov_type="robust")
print(result_unique.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:         unique_ratings   R-squared:                        0.1150
Estimator:                   PanelOLS   R-squared (Between):              0.3950
No. Observations:               40142   R-squared (Within):               0.1150
Date:                Tue, Jun 30 2026   R-squared (Overall):              0.3971
Time:                        12:56:44   Log-likelihood                -1.799e+04
Cov. Estimator:                Robust                                           
                                        F-statistic:                      360.85
Entities:                       29029   P-value                           0.0000
Avg Obs:                       1.3828   Distribution:                 F(4,11109)
Min Obs:                       1.0000                                           
Max Obs:                       37.000   F-statistic (robust):             118.90
                            

/var/folders/n_/7ftmcsrs6ng_hzblgzjhb9040000gn/T/ipykernel_29802/2705198778.py:16: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

total_reviews, months_since_first_review

  result_unique = model_unique.fit(cov_type="robust")


In [42]:
model_var = PanelOLS.from_formula(
    """
    var_rating ~
    is_bundled +
    n_reviews +
    total_reviews +
    months_since_first_review +
    weekend +
    hour_fe
    """,
    data=reg_df,
    drop_absorbed=True
)

result_var = model_var.fit(cov_type="robust")
print(result_var.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:             var_rating   R-squared:                        0.5357
Estimator:                   PanelOLS   R-squared (Between):              0.5481
No. Observations:               40142   R-squared (Within):              -0.0511
Date:                Tue, Jun 30 2026   R-squared (Overall):              0.5357
Time:                        12:57:08   Log-likelihood                -5.348e+04
Cov. Estimator:                Robust                                           
                                        F-statistic:                      7717.7
Entities:                       29029   P-value                           0.0000
Avg Obs:                       1.3828   Distribution:                 F(6,40136)
Min Obs:                       1.0000                                           
Max Obs:                       37.000   F-statistic (robust):             8383.1
                            

In [43]:
model_unique = PanelOLS.from_formula(
    """
    unique_ratings ~ 
    is_bundled +
    n_reviews +
    total_reviews +
    months_since_first_review +
    weekend +
    hour_fe
    """,
    data=reg_df,
    drop_absorbed=True
)

result_unique = model_unique.fit(cov_type="robust")
print(result_unique.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:         unique_ratings   R-squared:                        0.9015
Estimator:                   PanelOLS   R-squared (Between):              0.9062
No. Observations:               40142   R-squared (Within):              -0.0328
Date:                Tue, Jun 30 2026   R-squared (Overall):              0.9015
Time:                        12:57:27   Log-likelihood                -5.372e+04
Cov. Estimator:                Robust                                           
                                        F-statistic:                   6.123e+04
Entities:                       29029   P-value                           0.0000
Avg Obs:                       1.3828   Distribution:                 F(6,40136)
Min Obs:                       1.0000                                           
Max Obs:                       37.000   F-statistic (robust):          6.368e+04
                            